# Notebook 3 – BDI Architecture with LangChain + Gemini
## Customer Support Agent with Tool-Calling Agent

---

### 1. What you will learn

- How **Beliefs, Desires, Intentions** map to a real LangChain agent
- How `@tool` decorated functions are the **reactive execution steps**
- How `create_tool_calling_agent` + `AgentExecutor` is the **deliberative component**
- How the LLM decides the plan and tools execute it — same architecture as Notebook_2 and 3, now production-grade

Use case:
> **E-commerce Customer Support Agent**

## 2. Architecture: BDI mapped to LangChain

```
USER MESSAGE
      ↓
┌─────────────────────────────────────────────────────┐
│  DELIBERATIVE COMPONENT  (LangChain AgentExecutor)  │
│                                                     │
│  Gemini reasons via system prompt:                  │
│    Beliefs  → what facts are true?                  │
│    Desires  → what does the customer want?          │
│    Intention→ which strategy? (compensate /         │
│               inform / escalate)                    │
│                                                     │
│  Gemini then produces a PLAN = sequence of          │
│  tool calls it decides to make                      │
└──────────────┬──────────────────────────────────────┘
               ↓  (for each tool call in the plan)
┌─────────────────────────────────────────────────────┐
│  REACTIVE COMPONENT  (@tool functions)              │
│                                                     │
│  fetch_order()           ← look up order.csv        │
│  check_shipping_status() ← interpret status         │
│  offer_compensation()    ← apply discount           │
│  provide_order_info()    ← surface order details    │
│  escalate_to_human()     ← flag for human review    │
│                                                     │
│  Each tool: deterministic, no LLM, fast             │
└──────────────┬──────────────────────────────────────┘
               ↓
         FINAL RESPONSE
```

**How this maps to Notebook_2:**
| Notebook 2 | This notebook |
|---|---|
| `plan()` returns a list of strings | `AgentExecutor` decides the tool-call sequence |
| `step_registry[step](state)` | LangChain dispatches `@tool` by name |
| Shared `state` dict | Agent scratchpad (tool results visible to LLM) |
| `for step in plan_steps` | AgentExecutor loop over tool calls |

## 3. Setup

```bash
pip install langchain langchain-google-genai python-dotenv pandas
```

In [17]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# Load orders knowledge base once — shared across all tools
orders_df = pd.read_csv("order.csv")
print(f"Loaded {len(orders_df)} orders")
print(orders_df.head(3))

Loaded 30 orders
   order_id   user_name                 product_name      status        date
0      5001    John Doe               Wireless Mouse   Delivered  2024-08-15
1      5002  Jane Smith              Gaming Keyboard     Shipped  2024-08-20
2      5003    John Doe  Noise Cancelling Headphones  Processing  2024-08-21


## 4. Reactive Steps — `@tool` Functions

Each function below is a **reactive execution step** decorated with `@tool`.

- They are **deterministic** — same input always produces the same output
- They do **not call the LLM** — no reasoning, just action
- The LangChain agent decides **which tools to call and in what order** (the deliberative part)

The `docstring` of each tool is what the LLM reads to decide whether to use it.

In [18]:
from typing import Optional

@tool
def fetch_order(order_id: int) -> str:
    """Fetch full order details from the database given an order ID."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    return json.dumps(result.iloc[0].to_dict(), default=str)


@tool
def check_shipping_status(order_id: int) -> str:
    """Get a human-readable shipping status message for an order."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    status = result.iloc[0]["status"]
    messages = {
        "Delivered":  "Your order has been delivered.",
        "Shipped":    "Your order is currently on the way.",
        "Processing": "Your order is still being prepared and has not shipped yet.",
        "Cancelled":  "Your order has been cancelled.",
    }
    return messages.get(status, "Order status unknown.")


@tool
def offer_compensation(order_id: int) -> str:
    """Apply a 10% discount to the customer's account as compensation for an order issue."""
    result = orders_df[orders_df["order_id"] == order_id]
    name = result.iloc[0]["user_name"] if not result.empty else "the customer"
    return f"10% discount successfully applied to {name}'s account."


@tool
def provide_order_info(order_id: int) -> str:
    """Provide detailed order information: product name, status, and order date."""
    result = orders_df[orders_df["order_id"] == order_id]
    if result.empty:
        return f"No order found with ID {order_id}."
    r = result.iloc[0]
    return f"Order #{r['order_id']} -- {r['product_name']}, Status: {r['status']}, Placed on: {r['date']}."


@tool
def escalate_to_human(order_id: Optional[int] = None) -> str:
    """Escalate a complex or unresolved issue to the human support team. order_id is optional."""
    suffix = f" for order {order_id}" if order_id else ""
    return (
        f"The issue{suffix} has been escalated to the human support team. "
        "A representative will contact the customer within 24 hours."
    )


tools = [fetch_order, check_shipping_status, offer_compensation, provide_order_info, escalate_to_human]

## 5. Deliberative Component — BDI System Prompt

The system prompt instructs Gemini to reason in **BDI style** before deciding which tools to call.

This is the deliberative component:
- Gemini reads the message and infers Beliefs, Desires, Intention
- Based on Intention, it decides the sequence of tool calls (the plan)
- `AgentExecutor` then dispatches those tool calls one by one (the reactive execution)

The `agent_scratchpad` slot is where LangChain records tool results so Gemini can see them during reasoning — equivalent to our shared `state` dict in Notebook_2.

In [19]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a BDI (Belief-Desire-Intention) customer support agent for an e-commerce platform.

Before calling any tools, reason through three stages:
1. BELIEFS  — What facts can you infer from the customer's message?
2. DESIRES  — What does the customer want?
3. INTENTION — Choose ONE resolution strategy:
   - 'provide_information'  : customer wants order details or status
   - 'offer_compensation'   : customer is upset about a delay or problem
   - 'escalate'             : issue is complex or cannot be resolved with tools

Then execute your intention by calling tools in this order:
1. Always call fetch_order first to get real order data
2. Call check_shipping_status to get the delivery situation
3. Based on your intention:
   - provide_information → call provide_order_info
   - offer_compensation  → call offer_compensation
   - escalate            → call escalate_to_human

Finally, compose an empathetic, personalised response using all tool results.
Never invent order details — always use what the tools return.
"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 6. Run the Agent

`verbose=True` on `AgentExecutor` shows every tool call — you can watch the deliberative plan being executed reactively, step by step.

In [20]:
# --- Angry customer asking for compensation ---
msg = "I have not received my order 5003 and this is unacceptable! I want compensation."

print(f"USER: {msg}")
print("="*60)

result = agent_executor.invoke({"input": msg, "chat_history": []})

print("="*60)
print(f"FINAL RESPONSE:\n{result['output']}")

USER: I have not received my order 5003 and this is unacceptable! I want compensation.


> Entering new AgentExecutor chain...

Invoking: `fetch_order` with `{'order_id': 5003.0}`


{"order_id": 5003, "user_name": "John Doe", "product_name": "Noise Cancelling Headphones", "status": "Processing", "date": "2024-08-21"}
Invoking: `check_shipping_status` with `{'order_id': 5003.0}`


Your order is still being prepared and has not shipped yet.
Invoking: `offer_compensation` with `{'order_id': 5003.0}`


10% discount successfully applied to John Doe's account.I am so sorry to hear that your order #5003 for Noise Cancelling Headphones, placed on 2024-08-21, has not yet arrived. I understand this is frustrating and unacceptable.

It looks like your order is still being prepared and has not shipped yet. As a token of our apology for the delay, I have successfully applied a 10% discount to your account.

We are working to get your order to you as soon as possible.

> Finished chain.
FINAL RESPON

In [21]:
# --- Multiple messages: Gemini picks a different intention and tool sequence each time ---
messages = [
    "Where is my order 5007? Just need a quick update.",
    "Order 5005 was cancelled without my consent! I'm furious and want this resolved.",
    "Hi, can you give me the full details for order 5020?",
]

for msg in messages:
    print(f"\n{'='*60}")
    print(f"USER: {msg}")
    print("="*60)
    result = agent_executor.invoke({"input": msg, "chat_history": []})
    print(f"\nFINAL RESPONSE:\n{result['output']}")


USER: Where is my order 5007? Just need a quick update.


> Entering new AgentExecutor chain...

Invoking: `fetch_order` with `{'order_id': 5007.0}`


{"order_id": 5007, "user_name": "Chris Johnson", "product_name": "Smartwatch", "status": "Processing", "date": "2024-08-22"}
Invoking: `check_shipping_status` with `{'order_id': 5007.0}`


Your order is still being prepared and has not shipped yet.
Invoking: `provide_order_info` with `{'order_id': 5007.0}`


Order #5007 -- Smartwatch, Status: Processing, Placed on: 2024-08-22.Hi Chris,

Thanks for reaching out!

Your order #5007 for the Smartwatch, placed on August 22, 2024, is currently processing and has not shipped yet.

We'll send you another update as soon as it's on its way!

> Finished chain.

FINAL RESPONSE:
Hi Chris,

Thanks for reaching out!

Your order #5007 for the Smartwatch, placed on August 22, 2024, is currently processing and has not shipped yet.

We'll send you another update as soon as it's on its way!

USER: Order 50

## 7. Key Observations

- **`verbose=True`** shows the plan being executed — each `> Entering tool...` line is one reactive step
- **The plan varies** per message — Gemini uses BDI reasoning in the system prompt to decide which tools to call
- **Tools are pure functions** — same input, same output, no LLM involved
- **`agent_scratchpad`** is the shared state — tool results accumulate there so Gemini can use them in later reasoning steps
- **`chat_history`** enables multi-turn memory

### Comparison across notebooks

| Concept | Notebook 2 | Notebook 3 (prev) | This notebook |
|---|---|---|---|
| Planner | `plan()` — hardcoded rules | Gemini → returns `plan` list | `AgentExecutor` — LLM decides tool calls |
| Executor | `step_registry` + for loop | `execute_plan()` loop | LangChain tool dispatcher |
| State | `state` dict | `state` dict | `agent_scratchpad` |
| Tools | Plain functions | Plain functions | `@tool` decorated |
| Production-ready | ❌ | Partial | ✅ |

## 8. Limitations of BDI Architecture

BDI is a significant improvement over purely reactive systems, but has three concrete weaknesses:

### Limitation 1 -- Everything goes through the LLM (no fast reflexes)
The BDI agent routes **every** message through Gemini -- even ones with obvious, instant answers like FAQ questions. Cost and latency are wasted on zero-reasoning tasks.

### Limitation 2 -- Abusive language has no instant reflex
When a customer uses abusive language, a well-designed agent should **immediately escalate** without any reasoning. But BDI sends the full message to Gemini, infers a BDI state, picks an intention (often `offer_compensation`), and calls tools anyway. The abusive trigger is buried inside the reasoning, not acted on as a reflex.

### Limitation 3 -- Blurred intent forces a single-intention choice
BDI must collapse a message into **one** intention. When a message has multiple conflicting goals ("I want status AND a refund"), the agent picks one and silently ignores the other. Complex real-world messages resist clean Belief/Desire/Intention separation.

**Run the cells below to observe each limitation live.**

In [22]:
import time

limitation_messages = [
    # Limitation 1: Static FAQ -- BDI wastes an LLM call on a zero-reasoning question
    "What is your refund policy?",

    # Limitation 2: Abusive language -- BDI reasons through it instead of instant escalation
    "You idiots! My order 5003 has still not arrived. This is outrageous!",

    # Limitation 3: Blurred intent -- two goals in one message; BDI can only pick one
    "My order 5007 still has not shipped. Can you tell me when it will arrive AND refund my shipping fee?",
]

bdi_agent_instance = GeminiBDIAgent()  # from the agent cell above

for msg in limitation_messages:
    print(f"\n{'='*60}")
    print(f"USER: {msg}")
    print("="*60)
    start = time.time()
    result = agent_executor.invoke({"input": msg, "chat_history": []})
    elapsed = time.time() - start
    print(f"\nFINAL RESPONSE:\n{result['output']}")
    print(f"\n[TIME TAKEN: {elapsed:.2f}s -- Gemini was called regardless of message complexity]")


USER: What is your refund policy?


> Entering new AgentExecutor chain...

Invoking: `escalate_to_human` with `{}`


The issue has been escalated to the human support team. A representative will contact the customer within 24 hours.I understand you're asking about our refund policy. Unfortunately, I don't have the ability to provide details on that. I've escalated your request to our human support team, and a representative will contact you within 24 hours to assist you further.

> Finished chain.

FINAL RESPONSE:
I understand you're asking about our refund policy. Unfortunately, I don't have the ability to provide details on that. I've escalated your request to our human support team, and a representative will contact you within 24 hours to assist you further.

[TIME TAKEN: 3.33s -- Gemini was called regardless of message complexity]

USER: You idiots! My order 5003 has still not arrived. This is outrageous!


> Entering new AgentExecutor chain...

Invoking: `fetch_order` with `{'ord

### What to observe in the output above

| Message | What BDI does | What it SHOULD do |
|---|---|---|
| Refund policy question | Calls Gemini, reasons through B/D/I, calls tools | Return static text instantly -- no LLM needed |
| Abusive + order query | Picks `offer_compensation` or `provide_information` | Instantly escalate due to abusive language |
| Status + refund fee | Picks ONE intention -- either status or compensation | Handle both goals, not just one |

**Notice in the verbose output:** even the refund policy question triggers multiple LLM reasoning steps and tool calls. The agent cannot distinguish between "zero-reasoning needed" and "needs deep reasoning" -- it always pays the full cost.

**See Notebook_4 to see how the Layered Architecture fixes this.**